# Data Cleaning & Preparing

In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_excel("Dataset for Data Analytics.xlsx")
print(df.dtypes)



OrderID                    object
Date               datetime64[ns]
CustomerID                 object
Product                    object
Quantity                    int64
UnitPrice                 float64
ShippingAddress            object
PaymentMethod              object
OrderStatus                object
TrackingNumber             object
ItemsInCart                 int64
CouponCode                 object
ReferralSource             object
TotalPrice                float64
dtype: object


In [4]:
df.isnull().sum()

OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64

In [5]:
#  Handle missing values

df = df.dropna(subset=['OrderID', 'CustomerID', 'Product', 'Quantity', 'UnitPrice'])

# For less critical columns, fill with placeholders
df['ShippingAddress'] = df['ShippingAddress'].fillna('Unknown')
df['PaymentMethod'] = df['PaymentMethod'].fillna('Not Specified')
df['OrderStatus'] = df['OrderStatus'].fillna('Pending')
df['TrackingNumber'] = df['TrackingNumber'].fillna('No Tracking')
df['CouponCode'] = df['CouponCode'].fillna('None')
df['ReferralSource'] = df['ReferralSource'].fillna('Direct')

In [6]:
#  Fix data types (ensure they match your specs)
df['OrderID'] = df['OrderID'].astype(str)
df['CustomerID'] = df['CustomerID'].astype(str)
df['Product'] = df['Product'].astype(str)
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce').fillna(0).astype(int)
df['UnitPrice'] = pd.to_numeric(df['UnitPrice'], errors='coerce').fillna(0)
df['TotalPrice'] = pd.to_numeric(df['TotalPrice'], errors='coerce').fillna(0)
df['ItemsInCart'] = pd.to_numeric(df['ItemsInCart'], errors='coerce').fillna(0).astype(int)


In [7]:
#  Remove duplicate rows
df = df.drop_duplicates(subset=['OrderID'])

# 5. Validate and correct logical inconsistencies
# TotalPrice should equal Quantity * UnitPrice 
df['CalculatedTotal'] = df['Quantity'] * df['UnitPrice']
df.loc[abs(df['TotalPrice'] - df['CalculatedTotal']) > 0.01, 'TotalPrice'] = df['CalculatedTotal']
df = df.drop(columns=['CalculatedTotal'])

# Quantity and UnitPrice should be positive
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] >= 0)]

In [8]:
# Standardize text columns
df['OrderStatus'] = df['OrderStatus'].str.strip().str.title()
df['PaymentMethod'] = df['PaymentMethod'].str.strip().str.lower()
df['ReferralSource'] = df['ReferralSource'].str.strip().str.lower()

In [10]:
# Remove outliers
if len(df) > 0:
    mean_price = df['TotalPrice'].mean()
    std_price = df['TotalPrice'].std()
    df = df[(df['TotalPrice'] >= mean_price - 3*std_price) & 
            (df['TotalPrice'] <= mean_price + 3*std_price)]

In [11]:
# Last check
print("\nCleaned shape:", df.shape)
print("\nData types after cleaning:\n", df.dtypes)
print("\nMissing values after cleaning:\n", df.isnull().sum())

# Save cleaned dataset
df.to_excel('cleaned_dataset.xlsx', index=False)


Cleaned shape: (1200, 14)

Data types after cleaning:
 OrderID                    object
Date               datetime64[ns]
CustomerID                 object
Product                    object
Quantity                    int64
UnitPrice                 float64
ShippingAddress            object
PaymentMethod              object
OrderStatus                object
TrackingNumber             object
ItemsInCart                 int64
CouponCode                 object
ReferralSource             object
TotalPrice                float64
dtype: object

Missing values after cleaning:
 OrderID            0
Date               0
CustomerID         0
Product            0
Quantity           0
UnitPrice          0
ShippingAddress    0
PaymentMethod      0
OrderStatus        0
TrackingNumber     0
ItemsInCart        0
CouponCode         0
ReferralSource     0
TotalPrice         0
dtype: int64
